# MP-Declare Constraint Mining

Mining MP-Declare constraints with data conditions from the Helpdesk event log using RuM's MINERful + MpEnhancer.

In [1]:
import sys
import os
from pathlib import Path

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

from src.interpretability.perturbation_methods import csv_to_xes, discover_mpdeclare

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:163: DeprecationWarning: module 'sre_parse' is deprecated
  import sre_parse
/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:164: DeprecationWarning: module 'sre_constants' is deprecated
  import sre_constants


In [2]:
csv_path = _current / 'data' / 'helpdesk.csv'
xes_path = _current / 'data' / 'helpdesk.xes'

csv_to_xes(csv_path, xes_path, case_id_col="Case ID", activity_col="Activity", timestamp_col="Complete Timestamp")
print("Converted CSV to XES")

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/pm4py/utils.py:1000: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn(


exporting log, completed traces ::   0%|          | 0/4580 [00:00<?, ?it/s]

Converted CSV to XES


In [3]:
# Mine MP-Declare constraints with data conditions
# data_conditions: "ACTIVATIONS" mines conditions on activation event attributes
#                  "CORRELATIONS" mines correlations between activation/target (may fail with timestamps)
#                  "NONE" mines activity-only constraints
constraints = discover_mpdeclare(xes_path, min_support=0.99, data_conditions='ACTIVATIONS')
print(f"Mined {len(constraints)} MP-Declare constraints")

log4j:WARN No appenders could be found for logger (minerful.miner.core.MinerFulKBCore).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


||||||||||||||||||||||||||||||||||||||||
2026-03-01 11:44:31,753 INFO    [main] task.discovery.mp_enhancer.MpEnhancer - MpEnhancer (588449070) started at: 1772361871751
2026-03-01 11:44:31,756 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Number of constraints to process: 5
2026-03-01 11:44:31,758 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 1: Constraint(supp=0.9963548): Response[Assign seriousness, Closed] | |


Mar 01, 2026 11:44:34 AM com.github.fommil.jni.JniNamer arch
Mar 01, 2026 11:44:34 AM com.github.fommil.netlib.ARPACK <clinit>
Mar 01, 2026 11:44:34 AM com.github.fommil.jni.JniNamer arch
Mar 01, 2026 11:44:34 AM com.github.fommil.netlib.ARPACK <clinit>


2026-03-01 11:44:46,557 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 2: Constraint(supp=1.0): Precedence[Require upgrade, Assign seriousness] | |
2026-03-01 11:44:46,576 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 3: Constraint(supp=0.9983799): Response[Assign seriousness, Resolve ticket] | |
2026-03-01 11:45:02,576 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 4: Constraint(supp=0.9979932): Response[Resolve ticket, Closed] | |
2026-03-01 11:45:23,045 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 5: Constraint(supp=1.0): Precedence[Closed, Resolve ticket] | |
2026-03-01 11:45:23,199 INFO    [main] task.discovery.mp_enhancer.MpEnhancer - MpEnhancer (588449070) completed at: 1772361923198 - total time: 51447
Mined 11 MP-Declare constraints


In [4]:
# Show constraints WITH data conditions
print("=" * 100)
print("MP-DECLARE CONSTRAINTS WITH DATA CONDITIONS")
print("=" * 100)
print()

with_data = [c for c in constraints if c.data_condition]
print(f"Found {len(with_data)} constraints with data conditions:\n")

for i, c in enumerate(with_data, 1):
    print(f"{i}. {c.template}[{c.activation}, {c.target}]")
    print(f"   Support: {c.support:.1%}")
    print(f"   Data condition: {c.data_condition}")
    print()

MP-DECLARE CONSTRAINTS WITH DATA CONDITIONS

Found 0 constraints with data conditions:



In [5]:
# Show ALL constraints
print("=" * 100)
print("ALL MINED CONSTRAINTS")
print("=" * 100)
print()

for i, c in enumerate(constraints, 1):
    print(f"{i:3}. {c}")

ALL MINED CONSTRAINTS

  1. Absence[Activity: "DUPLICATE" (supp=0.99978167)] (support=100.0%)
  2. Absence[Activity: "INVALID" (supp=0.99956334)] (support=100.0%)
  3. Absence[Activity: "RESOLVED" (supp=0.99956334)] (support=100.0%)
  4. Absence[Activity: "Resolve SW anomaly" (supp=0.9982533)] (support=99.8%)
  5. Absence[Activity: "Schedule intervention" (supp=0.9989083)] (support=99.9%)
  6. Absence[Activity: "VERIFIED" (supp=0.999345)] (support=99.9%)
  7. Response[Activity: "Assign seriousness" (supp=0.9963548), Activity: "Closed" (supp=0.9963548)] (support=99.6%)
  8. Precedence[Activity: "Assign seriousness" (supp=1.0), Activity: "Require upgrade" (supp=1.0)] (support=100.0%)
  9. Response[Activity: "Assign seriousness" (supp=0.9983799), Activity: "Resolve ticket" (supp=0.9983799)] (support=99.8%)
 10. Response[Activity: "Resolve ticket" (supp=0.9979932), Activity: "Closed" (supp=0.9979932)] (support=99.8%)
 11. Precedence[Activity: "Resolve ticket" (supp=1.0), Activity: "Closed"

# REVISED+ Counterfactual Explanations

Generate counterfactual prefixes using the REVISED+ orchestrator:
- VAE trained on **random-length prefixes** (learns the prefix manifold)
- **Two-tier plausibility**: prefix-safe constraints for search penalty, all constraints for validity gate
- Latent space elite-sampling search

In [6]:
import torch

# Add src to path for event_log_loader module (needed by torch.load)
import sys
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# --- Load dataset ---
data_path = _current / 'encoded_data' / 'test_philipp' / 'helpdesk_all_5_test.pkl'
dataset = torch.load(data_path, weights_only=False)

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f"Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}")

# --- Load trained prediction model ---
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

model_path = _current / 'src' / 'notebooks' / 'training_variational_dropout' / 'Helpdesk' / 'Helpdesk_full_grad_norm_new_4layer.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(model_path), dropout=0.0)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Dataset: 6077 sequences, 12 cat, 4 num, seq_len=18
Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Resource', 24, {'EOS': 1, 'Value 1': 2, 'Value 10': 3, 'Value 11': 4, 'Value 12': 5, 'Value 13': 6, 'Value 14': 7, 'Value 15': 8, 'Value 16': 9, 'Value 17': 10, 'Value 18': 11, 'Value 19': 12, 'Value 2': 13, 'Value 20': 14, 'Value 21': 15, 'Value 22': 16, 'Value 3': 17, 'Value 4': 18, 'Value 5': 19, 'Value 6': 20, 'Value 7': 21, 'Value 8': 22, 'Value 9': 23}), ('VariantIndex', 175, {'1.0': 1, '10.0': 2, '100.0': 3, '103.0': 4, '104.0': 5, '107.0': 6, '109.0': 7, '11.0': 8, '110.0': 9, '112.0': 10, '113.0': 11, '114.0': 12, '115.0': 13, '117.0': 14, '118.0': 15, '12.0': 16, '120.0': 17, '122.0': 18, '123

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/philippeichhorn/.local/share/v

In [7]:
# --- Setup TensorDecoder and activity names ---
from src.interpretability.utils.tensor_decoder import TensorDecoder

decoder = TensorDecoder(dataset)

# Build activity_names list: index -> name (for the REVISED+ orchestrator)
activity_idx_to_label = decoder.idx_to_label['Activity']
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]

print(f"Activity vocabulary ({len(activity_names)}):")
for i, name in enumerate(activity_names):
    print(f"  {i}: {name}")

Activity vocabulary (16):
  0: <pad>
  1: Assign seriousness
  2: Closed
  3: Create SW anomaly
  4: DUPLICATE
  5: EOS
  6: INVALID
  7: Insert ticket
  8: RESOLVED
  9: Require upgrade
  10: Resolve SW anomaly
  11: Resolve ticket
  12: Schedule intervention
  13: Take in charge ticket
  14: VERIFIED
  15: Wait


In [8]:
# --- Create and fit REVISED+ ---
# This trains the VAE on random-length prefixes and mines Declare constraints.
# The VAE is saved to disk after first training and loaded on subsequent runs.
from src.interpretability.perturbation_methods import RevisedPlus, RevisedPlusConfig, create_revised_plus_for_model

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

config = RevisedPlusConfig(
    vae_epochs=100,
    vae_kl_weight=0.1,
    declare_min_support=0.9,
    n_candidates_per_round=200,
    n_search_rounds=5,
    top_k=5,
    min_plausibility=0.0,  # no hard filter for now
    device=device,
)

vae_path = str(_current / 'encoded_data' / 'test_philipp' / 'helpdesk_vae.pkl')

rp = create_revised_plus_for_model(
    model=model,
    dataset=dataset,
    activity_names=activity_names,
    config=config,
    vae_path=vae_path,
)
print(f"\nREVISED+ ready:")
print(f"  VAE parameters: {sum(p.numel() for p in rp.vae.parameters()):,}")
print(f"  VAE path: {vae_path}")
print(f"  All constraints: {len(rp.all_constraints)}")
print(f"  Prefix-safe constraints: {len(rp.prefix_safe_constraints)}")

Loading VAE from /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/encoded_data/test_philipp/helpdesk_vae.pkl
Extracting activity sequences...
Mined 1088 constraints (507 prefix-safe)

REVISED+ ready:
  VAE parameters: 278,487
  VAE path: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/encoded_data/test_philipp/helpdesk_vae.pkl
  All constraints: 1088
  Prefix-safe constraints: 507


In [9]:
# --- Show mined Declare constraints with human-readable names ---
print("PREFIX-SAFE constraints (used in search penalty):")
print("-" * 60)
for c in sorted(rp.prefix_safe_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

print(f"\nNOT prefix-safe constraints (used in validity gate):")
print("-" * 60)
not_safe = rp.all_constraints - rp.prefix_safe_constraints
for c in sorted(not_safe, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

PREFIX-SAFE constraints (used in search penalty):
------------------------------------------------------------
  absence(Create SW anomaly, n=1)
  absence(INVALID, n=1)
  absence(Insert ticket, n=1)
  absence(RESOLVED, n=1)
  absence(Require upgrade, n=1)
  absence(Resolve SW anomaly, n=1)
  absence(Schedule intervention, n=1)
  absence(VERIFIED, n=1)
  alternate_precedence(Assign seriousness, Closed)
  alternate_precedence(Assign seriousness, Create SW anomaly)
  alternate_precedence(Assign seriousness, INVALID)
  alternate_precedence(Assign seriousness, Insert ticket)
  alternate_precedence(Assign seriousness, RESOLVED)
  alternate_precedence(Assign seriousness, Require upgrade)
  alternate_precedence(Assign seriousness, Resolve SW anomaly)
  alternate_precedence(Assign seriousness, Schedule intervention)
  alternate_precedence(Assign seriousness, VERIFIED)
  alternate_precedence(Assign seriousness, Wait)
  alternate_precedence(Closed, Create SW anomaly)
  alternate_precedence(Closed

In [10]:
# --- Generate counterfactual explanation for a single prefix ---
# Pick a test case that is still in-progress (no EOS), so the model is less
# certain and minimal counterfactuals are more interesting.
# We scan for a case with moderate prediction confidence.
import numpy as np

# Find a good candidate: in-progress prefix with uncertain prediction
best_idx, best_prob = None, 1.0
for i in range(min(200, len(dataset))):
    cat_t, num_t, _ = dataset[i]
    act = cat_t[0]
    # Skip completed traces (contain EOS = index 5)
    if (act == 5).any():
        continue
    cat_in = [c.unsqueeze(0) for c in cat_t]
    num_in = [n.unsqueeze(0) for n in num_t]
    with torch.no_grad():
        preds = model((cat_in, num_in))[0]
        logits = preds[0]["Activity_mean"][0]
        p = torch.softmax(logits, dim=-1)
        top_p = p.max().item()
    if 0.4 < top_p < best_prob:
        best_idx, best_prob = i, top_p

test_idx = best_idx if best_idx is not None else 86
print(f"Selected test_idx={test_idx} (top_p={best_prob:.3f})")

cat_tuple, num_tuple, case_id = dataset[test_idx]

# Show original prefix
print(f"\nCase: {case_id}")
df_orig = decoder.decode_sequence(cat_tuple, num_tuple, case_id=case_id)
display(df_orig)

# Prepare inputs for explain()
cat_tensors = [c.unsqueeze(0) for c in cat_tuple]
num_tensors = [n.unsqueeze(0) for n in num_tuple]

# Generate explanation (target_class=None means any different prediction)
explanation = rp.explain(cat_tensors, num_tensors, target_class=None)
print(explanation)

Selected test_idx=51 (top_p=0.495)

Case: Case 1061


,Activity,Resource,Variant index,seriousness,customer,product,responsible_section,seriousness_2,service_level,service_type,support_section,workgroup,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day,Case ID
0,Assign seriousness,Value 8,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,0.000000e+00,9.633033e+05,1.000000e+00,32105.000000,Case 1061
1,Assign seriousness,Value 8,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,1.600000e+01,1.606250e+01,1.000000e+00,32120.998047,Case 1061
2,Take in charge ticket,Value 13,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,5.201629e+05,5.201470e+05,-2.384186e-07,33868.000000,Case 1061
3,Wait,Value 13,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,2.521179e+06,2.001016e+06,2.000000e+00,47683.996094,Case 1061
4,Resolve ticket,Value 13,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,3.198710e+06,6.775310e+05,3.000000e+00,34015.000000,Case 1061


Original: Resolve ticket (p=0.495), prefix_len=5
REVISED+ Counterfactual Explanation
Original: Resolve ticket (idx=11, p=0.495)
Prefix length: 5 events
Target: any different class
Constraints: 507 prefix-safe, 1088 total
Search: 1000 evaluated, 664 valid, 17.7s

Top 5 counterfactuals:
  [1] -> Closed (p=0.904) | prox=20.427 sparse=5 feas=0.287 plaus_def=1.00 plaus_opt=0.98 score=1.2841
  [2] -> Closed (p=0.905) | prox=20.214 sparse=5 feas=0.448 plaus_def=1.00 plaus_opt=0.98 score=1.2975
  [3] -> EOS (p=0.998) | prox=24.316 sparse=5 feas=0.237 plaus_def=1.00 plaus_opt=0.99 score=1.3276
  [4] -> Closed (p=0.879) | prox=25.556 sparse=5 feas=0.460 plaus_def=0.99 plaus_opt=0.97 score=1.3714
  [5] -> Closed (p=0.901) | prox=29.509 sparse=6 feas=0.715 plaus_def=1.00 plaus_opt=0.98 score=1.3789


In [11]:
# --- Compare original vs counterfactual prefixes ---
if explanation.counterfactuals:
    best = explanation.get_best()

    print("=" * 80)
    print("ORIGINAL PREFIX")
    print(f"Prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print(f"Prefix length: {explanation.prefix_len} events")
    print("=" * 80)
    display(df_orig)

    print(f"\n{'=' * 80}")
    print("BEST COUNTERFACTUAL PREFIX")
    print(f"Prediction: {best.counterfactual_prediction_name} (p={best.counterfactual_probability:.3f})")
    cf_prefix_len = len(best.activity_sequence)
    print(f"Prefix length: {cf_prefix_len} events (delta={cf_prefix_len - explanation.prefix_len:+d})")
    print(f"Proximity: {best.proximity:.3f}  Sparsity: {best.sparsity}")
    print(f"Feasibility: {best.feasibility:.3f}")
    print(f"Plausibility (definite): {best.plausibility_definite:.2f}")
    print(f"Plausibility (optimistic): {best.plausibility_optimistic:.2f}")
    print(f"Combined score: {best.combined_score:.4f}")
    print("=" * 80)

    # Decode the counterfactual
    cf_cat = tuple(best.cat_sequence)
    cf_num = tuple(best.num_sequence[:, i] for i in range(best.num_sequence.shape[1])) if best.num_sequence is not None else num_tuple
    df_cf = decoder.decode_sequence(cf_cat, cf_num, skip_padding=False)
    display(df_cf)
else:
    print("No counterfactuals found. Try increasing n_search_rounds or noise_scale.")

ORIGINAL PREFIX
Prediction: Resolve ticket (p=0.495)
Prefix length: 5 events


,Activity,Resource,Variant index,seriousness,customer,product,responsible_section,seriousness_2,service_level,service_type,support_section,workgroup,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day,Case ID
0,Assign seriousness,Value 8,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,0.000000e+00,9.633033e+05,1.000000e+00,32105.000000,Case 1061
1,Assign seriousness,Value 8,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,1.600000e+01,1.606250e+01,1.000000e+00,32120.998047,Case 1061
2,Take in charge ticket,Value 13,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,5.201629e+05,5.201470e+05,-2.384186e-07,33868.000000,Case 1061
3,Wait,Value 13,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,2.521179e+06,2.001016e+06,2.000000e+00,47683.996094,Case 1061
4,Resolve ticket,Value 13,17.0,Value 1,Value 231,Value 12,Value 1,Value 1,Value 2,Value 1,Value 1,Value 1,3.198710e+06,6.775310e+05,3.000000e+00,34015.000000,Case 1061



BEST COUNTERFACTUAL PREFIX
Prediction: Closed (p=0.904)
Prefix length: 5 events (delta=+0)
Proximity: 20.427  Sparsity: 5
Feasibility: 0.287
Plausibility (definite): 1.00
Plausibility (optimistic): 0.98
Combined score: 1.2841


,Activity,Resource,Variant index,seriousness,customer,product,responsible_section,seriousness_2,service_level,service_type,support_section,workgroup,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day
0,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,1.109730e+06,1.077318e+06,2.535683,32979.285156
1,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,1.313320e+05,4.048428e+05,3.038005,29572.035156
2,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-7.472294e+05,-1.888624e+05,2.956426,39720.664062
3,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-1.008029e+06,-1.794516e+05,2.816238,43998.199219
4,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-1.018559e+06,-4.278688e+04,2.669814,44880.714844
5,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-9.594452e+05,6.327538e+04,2.552911,44818.515625
6,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-8.797938e+05,1.640451e+05,2.453350,44346.070312
7,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-7.839970e+05,2.771936e+05,2.358061,43550.203125
8,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-6.710479e+05,4.052318e+05,2.260248,42348.898438
9,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,<pad>,-5.420280e+05,5.401425e+05,2.155302,40528.488281


In [12]:
# --- Show all top-k counterfactuals ---
import pandas as pd

if explanation.counterfactuals:
    rows = []
    for i, cf in enumerate(explanation.counterfactuals):
        rows.append({
            'rank': i + 1,
            'prediction': cf.counterfactual_prediction_name,
            'probability': f"{cf.counterfactual_probability:.3f}",
            'prefix_len': len(cf.activity_sequence),
            'activities': ' -> '.join(activity_names[a] for a in cf.activity_sequence),
            'proximity': f"{cf.proximity:.2f}",
            'sparsity': cf.sparsity,
            'feasibility': f"{cf.feasibility:.3f}",
            'plaus_def': f"{cf.plausibility_definite:.2f}",
            'plaus_opt': f"{cf.plausibility_optimistic:.2f}",
            'score': f"{cf.combined_score:.4f}",
        })

    df_cfs = pd.DataFrame(rows)
    print(f"Original: {' -> '.join(activity_names[a] for a in explanation.original_activity_sequence)}")
    print(f"Original prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print()
    display(df_cfs)

Original: Assign seriousness -> Assign seriousness -> Take in charge ticket -> Wait -> Resolve ticket
Original prediction: Resolve ticket (p=0.495)



,rank,prediction,probability,prefix_len,activities,proximity,sparsity,feasibility,plaus_def,plaus_opt,score
0,1,Closed,0.904,5,Assign seriousness -> Take in charge ticket ->...,20.43,5,0.287,1.00,0.98,1.2841
1,2,Closed,0.905,5,Assign seriousness -> Assign seriousness -> Ta...,20.21,5,0.448,1.00,0.98,1.2975
2,3,EOS,0.998,5,Assign seriousness -> Take in charge ticket ->...,24.32,5,0.237,1.00,0.99,1.3276
3,4,Closed,0.879,5,Assign seriousness -> Assign seriousness -> Ta...,25.56,5,0.460,0.99,0.97,1.3714
4,5,Closed,0.901,6,Assign seriousness -> Assign seriousness -> Ta...,29.51,6,0.715,1.00,0.98,1.3789
